# Table 3: environmental exposure
The saved Table 3 results identify the paper's corridor values as the bounded transform `TightCorridorIndex / (1 + TightCorridorIndex)`, excluding frames with no visible floor for this metric only. The paper's supplementary equation omits this transform. This notebook preserves the computation that matches Table 3. Participant exposures are means of analysed frames, followed by equal-weight participant group summaries and Welch tests.


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
OUT = Path("results")
OUT.mkdir(exist_ok = True)
from scipy.stats import ttest_ind

METRICS = {
    'VisibleOpenFloorRatio': 'Open-floor ratio',
    'ClutterIndex':'Clutter index',
    'BoundedCorridorIndex':'Corridor occupancy',
    'StairsPresent':'Stairs',
    'PathObstructionIndex':'Path obstruction'
}

keys = ['Participant','BoutIndex']

In [5]:

p = pd.read_parquet('derived/participants.parquet',columns = ['Participant','TestStart'])
p = p.loc[~p.Participant.str.startswith('TOPAL')].copy()

p['TestStart'] = pd.to_datetime(p.TestStart)
p['Group'] = np.where(p.Participant.str.contains('PD'),'PwP','Control')
b = pd.read_parquet('derived/bouts.parquet',columns = keys + ['BoutStart','BoutEnd']).merge(p, on = 'Participant', validate = 'many_to_one')

for col in ['BoutStart','BoutEnd']:
    b[col] = pd.to_datetime(b[col],format = '%d/%m/%y %H:%M:%S')

start = (b.BoutStart - b.TestStart).dt.total_seconds()/60
end = (b.BoutEnd - b.TestStart).dt.total_seconds()/60

b['Condition'] = pd.Series(pd.NA,index = b.index,dtype = 'string')
for label,mask in {'Scripted':end.lt(2),'Semi-scripted':start.gt(5)&end.lt(20),'Free-living':start.gt(60)}.items():
    b.loc[mask,'Condition'] = label

In [6]:
f = pd.read_parquet('derived/frames.parquet',columns = keys+['SkippedHeavyInference']+['TightCorridorIndex' if c == 'BoundedCorridorIndex' else c for c in METRICS])
f = f.loc[f.SkippedHeavyInference.eq(False)].merge(b[keys+['Condition']],on = keys,validate = 'many_to_one').dropna(subset = ['Condition'])
raw = f['TightCorridorIndex'].where(f.VisibleOpenFloorRatio.gt(0) & np.isfinite(f.VisibleOpenFloorRatio))
f['BoundedCorridorIndex'] = raw / (1+raw)
participant_exposure = f.groupby(['Condition','Participant'])[list(METRICS)].mean().reset_index().merge(p[['Participant','Group']],on = 'Participant',validate = 'many_to_one')

rows = []
for condition in ['Scripted','Semi-scripted','Free-living']:
    sub = participant_exposure.loc[participant_exposure.Condition.eq(condition)]
    for column,label in METRICS.items():
        all_values = sub[column].dropna()
        a = sub.loc[sub.Group.eq('PwP'),column].dropna().to_numpy(float)
        c = sub.loc[sub.Group.eq('Control'),column].dropna().to_numpy(float)
        rows.append({
            'Condition':condition,
            'Metric':label,
            'Source_column':column,
            'Mean':all_values.mean(),
            'SD':all_values.std(),
            'Minimum':all_values.min(),
            'Maximum':all_values.max(),
            'PwP':a.mean(),
            'Control':c.mean(),
            'Difference':a.mean()-c.mean(),
            'p':ttest_ind(a,c,equal_var = False).pvalue,
            'N_PwP':len(a),'N_Control':len(c)
        })
        
table3 = pd.DataFrame(rows)
display(table3.round(3))
table3.to_csv(OUT/'table3.csv',index = False)
participant_exposure.to_csv(OUT/'table3_participant_exposure.csv',index = False)

,Condition,Metric,Source_column,Mean,SD,Minimum,Maximum,PwP,Control,Difference,p,N_PwP,N_Control
0,Scripted,Open-floor ratio,VisibleOpenFloorRatio,0.331,0.099,0.105,0.581,0.353,0.303,0.050,0.223,14,11
1,Scripted,Clutter index,ClutterIndex,0.409,0.100,0.340,0.769,0.418,0.397,0.022,0.586,14,11
2,Scripted,Corridor occupancy,BoundedCorridorIndex,0.318,0.085,0.146,0.524,0.310,0.328,-0.017,0.606,14,11
3,Scripted,Stairs,StairsPresent,0.013,0.051,0.000,0.257,0.020,0.003,0.017,0.367,14,11
4,Scripted,Path obstruction,PathObstructionIndex,0.123,0.116,0.007,0.485,0.096,0.156,-0.059,0.238,14,11
5,Semi-scripted,Open-floor ratio,VisibleOpenFloorRatio,0.462,0.180,0.002,0.765,0.467,0.458,0.009,0.886,17,17
6,Semi-scripted,Clutter index,ClutterIndex,0.324,0.144,0.122,1.078,0.293,0.355,-0.062,0.222,17,17
7,Semi-scripted,Corridor occupancy,BoundedCorridorIndex,0.277,0.143,0.000,0.835,0.286,0.268,0.018,0.721,17,17
8,Semi-scripted,Stairs,StairsPresent,0.162,0.089,0.000,0.353,0.144,0.181,-0.037,0.229,17,17
9,Semi-scripted,Path obstruction,PathObstructionIndex,0.142,0.125,0.018,0.547,0.142,0.141,0.001,0.979,17,17
